# Silver Layer — CRM Sales Details
Clean and normalize `crm_sales_details`.

## Setup Connection

In [ ]:
import os
from dotenv import load_dotenv
from clickzetta.zettapark.session import Session

load_dotenv()
session = Session.builder.configs({
    "username":  os.environ["CLICKZETTA_USERNAME"],
    "password":  os.environ["CLICKZETTA_PASSWORD"],
    "service":   os.environ["CLICKZETTA_SERVICE"],
    "instance":  os.environ["CLICKZETTA_INSTANCE"],
    "workspace": os.environ["CLICKZETTA_WORKSPACE"],
    "schema":    os.environ["CLICKZETTA_SCHEMA"],
    "vcluster":  os.environ["CLICKZETTA_VCLUSTER"],
}).create()
SCHEMA = os.environ["CLICKZETTA_SCHEMA"]
VOLUME = os.environ.get("CLICKZETTA_VOLUME", "medallion_vol")

## Read Bronze Table

In [ ]:
df = session.table(f"{SCHEMA}.crm_sales_details")

## Silver Transformations

### Trimming

In [ ]:
from clickzetta.zettapark.types import StringType, DateType
from clickzetta.zettapark import functions as F

for field in df.schema.fields:
    if isinstance(field.datatype, StringType):
        df = df.with_column(field.name, F.trim(F.col(field.name)))

### Date Parsing (yyyyMMdd integer → DATE)

In [ ]:
for date_col in ["sls_order_dt", "sls_ship_dt", "sls_due_dt"]:
    df = df.with_column(
        date_col,
        F.when(
            (F.col(date_col) == 0) | (F.length(F.col(date_col).cast("string")) != 8),
            F.lit(None).cast(DateType())
        ).otherwise(F.to_date(F.col(date_col).cast("string"), "yyyyMMdd"))
    )

### Price Cleanup

In [ ]:
df = df.with_column(
    "sls_price",
    F.when(
        F.col("sls_price").is_null() | (F.col("sls_price") <= 0),
        F.when(F.col("sls_quantity") != 0,
               F.col("sls_sales") / F.col("sls_quantity"))
         .otherwise(F.lit(None))
    ).otherwise(F.col("sls_price"))
)

### Rename Columns

In [ ]:
RENAME_MAP = {
    "sls_ord_num":  "order_number",
    "sls_prd_key":  "product_number",
    "sls_cust_id":  "customer_id",
    "sls_order_dt": "order_date",
    "sls_ship_dt":  "ship_date",
    "sls_due_dt":   "due_date",
    "sls_sales":    "sales_amount",
    "sls_quantity": "quantity",
    "sls_price":    "price",
}
for old, new in RENAME_MAP.items():
    df = df.with_column_renamed(old, new)

## Sanity Check

In [ ]:
df.limit(10).show()

## Write Silver Table

In [ ]:
df.write.save_as_table(f"{SCHEMA}.crm_sales", mode="overwrite")
print("crm_sales OK")

## Verify

In [ ]:
session.table(f"{SCHEMA}.crm_sales").limit(5).show()